In [1]:
from pathlib import Path
import os

from dotenv import load_dotenv
from huggingface_hub import HfApi, hf_hub_download, login, whoami
import pandas as pd
from tqdm import tqdm

e:\miniconda\envs\sara-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()  
TOKEN = os.getenv("HF_TOKEN")

login(token=TOKEN, add_to_git_credential=False)
print("✅ Logged in as:", whoami(token=TOKEN)["name"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Logged in as: sara-estilaei


In [3]:
load_dotenv(Path("..") / ".env")
hf_token = os.getenv("HF_TOKEN")

raw_dir = Path("../data/cv/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

src_cv_dir = Path("../src/cv")
src_cv_dir.mkdir(parents=True, exist_ok=True)

csv_files = [
    "sample_submission.csv",
    "solution.csv",
    "time_to_accident_test_map.csv",
]

eval_files = [
    "evaluate_submission.py",
]

print("Downloading metadata and evaluation files...")

for filename in csv_files:
    hf_hub_download(
        repo_id="nexar-ai/nexar_collision_prediction",
        repo_type="dataset",
        filename=filename,
        local_dir=raw_dir,
        token=hf_token,
    )
    print(f"✅ Downloaded (CSV): {filename} -> data/raw/")

for filename in eval_files:
    hf_hub_download(
        repo_id="nexar-ai/nexar_collision_prediction",
        repo_type="dataset",
        filename=filename,
        local_dir=src_cv_dir,
        token=hf_token,
    )
    print(f"✅ Downloaded (Script): {filename} -> src/cv/")

✅ Downloaded (CSV): sample_submission.csv -> data/raw/
✅ Downloaded (CSV): solution.csv -> data/raw/
✅ Downloaded (CSV): time_to_accident_test_map.csv -> data/raw/
✅ Downloaded (Script): evaluate_submission.py -> src/cv/


In [4]:
raw_dir = Path("../data/cv/raw")

df_solution = pd.read_csv(raw_dir / "solution.csv")

print(f"Total samples: {len(df_solution)}")
print("\nColumns in file:")
print(df_solution.columns.tolist())

print("\nFirst 5 rows:")
df_solution.head()

Total samples: 1344

Columns in file:
['id', 'target', 'Usage', 'group']

First 5 rows:


,id,target,Usage,group
0,204,1,Public,0
1,30,1,Public,0
2,146,1,Public,0
3,20,1,Public,0
4,511,1,Public,0


In [5]:
# Load environment variables from .env located in the project root
load_dotenv(Path("..") / ".env")
HF_TOKEN = os.getenv("HF_TOKEN")

api = HfApi()
repo_id = "nexar-ai/nexar_collision_prediction"

# List all files in the repository
all_files = api.list_repo_files(repo_id=repo_id, repo_type="dataset", token=HF_TOKEN)

# Filter video files by extension
video_files = [f for f in all_files if f.lower().endswith((".mp4", ".mov", ".avi"))]

print(f"Total video files in repo: {len(video_files)}")
print("Sample video paths:")
for f in video_files[:5]:
    print(" -", f)

Total video files in repo: 2844
Sample video paths:
 - test-private/negative/01047.mp4
 - test-private/negative/01053.mp4
 - test-private/negative/01055.mp4
 - test-private/negative/01057.mp4
 - test-private/negative/01063.mp4


In [6]:
import pandas as pd

raw_dir = Path("..") / "data" / "cv" / "raw"
df_solution = pd.read_csv(raw_dir / "solution.csv")
df_solution["target"] = df_solution["target"].astype(int)

# Pick a few balanced samples (e.g., 3 positive and 3 negative)
n_samples = 3
pos_samples = df_solution[df_solution["target"] == 1].head(n_samples)
neg_samples = df_solution[df_solution["target"] == 0].head(n_samples)

selected_df = pd.concat([pos_samples, neg_samples]).reset_index(drop=True)
selected_df[["id", "target", "Usage"]]

,id,target,Usage
0,204,1,Public
1,30,1,Public
2,146,1,Public
3,1877,0,Public
4,1898,0,Public
5,1367,0,Public


In [7]:
# Select 25 positive and 25 negative samples
n_per_class = 25

pos_samples = df_solution[df_solution["target"] == 1].head(n_per_class)
neg_samples = df_solution[df_solution["target"] == 0].head(n_per_class)

selected_df = pd.concat([pos_samples, neg_samples]).reset_index(drop=True)

print(f"Selected {len(selected_df)} videos:")
print(f"Positive: {(selected_df['target'] == 1).sum()}")
print(f"Negative: {(selected_df['target'] == 0).sum()}")

# Store the downloaded videos here
sample_dir = Path("../data/cv/raw/sample_videos")
sample_dir.mkdir(parents=True, exist_ok=True)

# Map video filenames to their repository paths
repo_file_map = {Path(path).name: path for path in video_files}

downloaded_records = []
missing_files = []

print("\nDownloading videos...")

for _, row in tqdm(selected_df.iterrows(), total=len(selected_df)):
    sample_id = int(row["id"])
    target = int(row["target"])
    filename = f"{sample_id:05d}.mp4"

    repo_path = repo_file_map.get(filename)

    if repo_path is None:
        missing_files.append(filename)
        continue

    # Check if file already exists locally to skip HF network checks completely
    target_file = sample_dir / repo_path

    if target_file.exists():
        local_path = target_file
        # print(f"Skipped (already exists): {filename}") # اختیاری برای لاگ
    else:
        local_path = hf_hub_download(
            repo_id=repo_id,
            filename=repo_path,
            repo_type="dataset",
            token=HF_TOKEN,
            local_dir=sample_dir,
        )

    downloaded_records.append(
        {
            "id": sample_id,
            "target": target,
            "filename": filename,
            "repo_path": repo_path,
            "local_path": str(local_path),
        }
    )

# Save an index of the downloaded videos
downloaded_df = pd.DataFrame(downloaded_records)
index_path = sample_dir / "videos_metadata.csv"
downloaded_df.to_csv(index_path, index=False)

print(f"\nDownloaded: {len(downloaded_records)} videos")
print(f"Saved under: {sample_dir.resolve()}")
print(f"Video index: {index_path.resolve()}")

if missing_files:
    print(f"\nFiles not found in the repository: {missing_files}")

Selected 50 videos:
Positive: 25
Negative: 25



100%|██████████| 50/50 [00:00<00:00, 8336.59it/s]


Downloaded: 50 videos
Saved under: C:\Users\98936\Desktop\incident-sentiment-analyzer\data\cv\raw\sample_videos
Video index: C:\Users\98936\Desktop\incident-sentiment-analyzer\data\cv\raw\sample_videos\videos_metadata.csv
